Every row contains columns such as:

Winner, Loser
WRank, LRank
WPts, LPts
AvgW, AvgL

Here, WRank does not mean “rank of player 1”. It means rank of the player who eventually won.

Therefore, you must not train a model directly using these columns.

For example, this would be completely wrong:

X = matches[["WRank", "LRank"]]
y = 1

Every row would have the winner in the first position, so the target would always be 1. The model would not be predicting anything.

You need to reformulate every match as:

Player1
Player2
Player1Rank
Player2Rank
Player1Points
Player2Points
...
Player1Wins = 0 or 1

For approximately half the matches, Player 1 should be the original winner. For the other half, Player 1 should be the original loser.

This is the first major operation you should perform before modelling.

FEATURES THAT SHOULD NOT BE USED FOR PREDITCTION 
The model should only receive information that was available before the match began.

| Column group                                     | What to do                                                                            |
| ------------------------------------------------ | ------------------------------------------------------------------------------------- |
| `Winner`, `Loser`                                | Use to construct the target and historical features, but not directly as model inputs |
| `W1–W5`, `L1–L5`                                 | Remove: set scores are known after the match                                          |
| `Wsets`, `Lsets`                                 | Remove: final result information                                                      |
| `Comment`                                        | Do not use as a predictor; it may reveal retirement, walkover, or completion          |
| `SourceFile`                                     | Keep for data analysis, but normally do not use as a feature                          |
| `WRank`, `LRank`                                 | Usable only after mapping them to Player 1 and Player 2                               |
| `WPts`, `LPts`                                   | Usable only after mapping                                                             |
| Betting odds                                     | Pre-match information, so usable after mapping                                        |
| `Surface`, `Court`, `Series`, `Round`, `Best of` | Valid pre-match information                                                           |

Using scores or match comments would be data leakage: the model would receive information produced by the event it is supposed to predict.

## 02 — Data Preparation

In this notebook, the yearly datasets are combined and cleaned.

The original Winner/Loser representation is transformed into a neutral
Player1/Player2 representation so that the position of a player does not
reveal the result of the match.

In [8]:
from pathlib import Path

import numpy as np 
import pandas as pd

In [9]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

PROCESSED_DATA_DIR.mkdir(parents = True, exist_ok=True) ##create processed data folder

print("Raw data folder:", RAW_DATA_DIR.resolve())
print("Processed data folder:", PROCESSED_DATA_DIR.resolve())

Raw data folder: C:\Users\User\Downloads\artificial_project\data\raw
Processed data folder: C:\Users\User\Downloads\artificial_project\data\processed


Why we select these columns

The older datasets contain columns such as:

EXW
EXL
LBW
LBL

while the 2025 dataset contains:

BFEW
BFEL

These columns are not consistently available across all years. Therefore, we will not include them in the combined table.

We keep bookmaker columns that are available throughout the period:

B365W, B365L
PSW, PSL
MaxW, MaxL
AvgW, AvgL

We are not deciding to use these columns in the main model yet. We are only preserving them.

Later, a sensible project structure will probably compare:

a model based on your own historical tennis features;
an optional experiment that also includes bookmaker odds.

This prevents the betting market information from replacing the feature-engineering part of your project.

Why scores are absent

We deliberately exclude:

W1–W5
L1–L5
Wsets
Lsets

These are produced during the match. They cannot be known when predicting the winner before the match begins.

Removing them now also prevents you from accidentally using them later.

In [10]:
years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

## I selected the columns that were in common to all years. 
## I choose them based on the result obtained in the data_understanding notebook
selected_columns = [
    "ATP", 
    "Location", 
    "Tournament", 
    "Date", 
    "Series", 
    "Court", 
    "Surface",
    "Round", 
    "Best of", 
    "Winner", 
    "Loser", 
    "WRank", 
    "LRank", 
    "WPts", 
    "LPts", 
    "Comment", 
    "B365W", 
    "B365L", 
    "PSW",
    "PSL",
    "MaxW",
    "MaxL",
    "AvgW",
    "AvgL" ]

yearly_frames = [] creates an empty Python list. Each cleaned yearly DataFrame will be added to this list.

The loop runs once for every year. 
During the first iteration: year = 2015. 
During the next iteration: year = 2016
and so on.

current_year.columns.str.strip()
strip() removes spaces at the beginning or end of a column name.
For example: "Winner " becomes: "Winner"
This protects us against subtle problems caused by accidental spaces.

The missing_column list checks whether every column we need is present.
I look at every selected column and keep only the names that are not present in the current dataset.
If a required column is missing, execution stops immediately and displays a meaningful error.
That is better than continuing with an incomplete dataset.

With .copy() , I creates a separate DataFrame rather than a potentially ambiguous view of the original one.
It prevents some common pandas warnings and nintended modifications.

SourceYear records which Excel file supplied the row.
For example, rows loaded from 2017.xlsx receive:
SourceYear = 2017
This is useful for checking the data, but it should not normally be supplied to the model.

pd.concat(...) places the yearly tables underneath one another.
ignore_index=True creates a new continuous row index:
0, 1, 2, 3, ...
The number of rows should initially be close to the total I already observed: 27,574.

In [11]:
yearly_frames = []

for year in years: 
    file_path = RAW_DATA_DIR / f"{year}.xlsx"
    
    current_year = pd.read_excel(file_path)
    
    #Remove accidental spaces from the column names 
    current_year.columns = current_year.columns.str.strip()
    
    missing_columns = [
        column
        for column in selected_columns 
        if column not in current_year.columns 
    ]
    
    if missing_columns: 
        raise ValueError(
            f"The {year} dataset is missing these columns: {missing_columns}"
        )
    
    current_year = current_year[selected_columns].copy()

    #Record the file from which every row originated.  
    current_year["SourceYear"] = year

    yearly_frames.append(current_year)  

matches = pd.concat(yearly_frames, ignore_index=True)

print("Combined dataset shape:", matches.shape)


c:\Users\User\Downloads\artificial_project\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\User\Downloads\artificial_project\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Combined dataset shape: (27574, 25)


In [13]:
print("Number of rows:", len(matches))
print("Number of columns:", len(matches.columns))

matches.head()

matches["SourceYear"].value_counts().sort_index()

Number of rows: 27574
Number of columns: 25


SourceYear
2015    2630
2016    2626
2017    2633
2018    2637
2019    2610
2020    1267
2021    2489
2022    2632
2023    2703
2024    2703
2025    2644
Name: count, dtype: int64

I need to standardize all the features. 
Different Excel files may represent the same kind of information differently.
For example, one ranking might be stored as the number: 54
while another file might accidentally store it as text: "54"
Machine-learning models require predictable data types.

errors="coerce" --> When pandas cannot convert a value, it replaces it with a missing value, represented by NaN or NaT.

For example: "unknown" cannot be converted to a number, so it becomes: NaN
This is preferable to silently leaving mixed text and numerical values in the same column.
** For dates, an invalid value becomes NaT, meaning “Not a Time”.

In [14]:
matches["Date"] = pd.to_datetime(
    matches["Date"], 
    errors="coerce"
)

text_columns = [
    "Location",
    "Tournament",
    "Series",
    "Court",
    "Surface",
    "Round",
    "Winner",
    "Loser",
    "Comment"
]

for column in text_columns: 
    matches[column] = matches[column].astype("string").str.strip()
    
numeric_columns = [
    "ATP",
    "Best of",
    "WRank",
    "LRank",
    "WPts",
    "LPts",
    "B365W",
    "B365L",
    "PSW",
    "PSL",
    "MaxW",
    "MaxL",
    "AvgW",
    "AvgL"  
]

for column in numeric_columns: 
    matches[column] = pd.to_numeric(
        matches[column], 
        errors="coerce"
    )

I have to check conversion problems that may occur. 
This tells me how many matches lack essential pre-match information.

**Do not worry if there are some missing rankings or ranking points. You already discovered three missing loser rankings and points in 2025, and older years may contain additional missing values.

In [15]:
essential_columns = [
    "Date", 
    "Winner", 
    "Loser", 
    "WRank", 
    "LRank", 
    "WPts",
    "LPts"
]

print(matches[essential_columns].isna().sum())

Date       0
Winner     0
Loser      0
WRank     11
LRank     58
WPts      10
LPts      58
dtype: int64


In [16]:
print(matches["Comment"].value_counts(dropna=False))

Comment
Completed       26602
Retired           794
Walkover          170
Awarded             5
Sched               1
Disqualified        1
Rrtired             1
Name: count, dtype: Int64


A walkover is not a normally played match.

A retirement may be caused by an injury or physical problem occurring during the match.

Your current features will describe player strength and previous performance, but they will not tell the model that a player is about to become injured.

For the first version of the project, completed matches therefore provide a clearer prediction problem.

This is a critical methodological choice that you can mention in the final report:

Retirements and walkovers were excluded because they do not represent normally completed matches and may be determined by events that are not captured by the available pre-match features.

In [17]:
rows_before_status_filter = len(matches)

matches = matches.loc[
    matches["Comment"].eq("Completed")
].copy()

rows_after_status_filter = len(matches)

print(
    "Matches removes because they were not completed:", 
    rows_before_status_filter - rows_after_status_filter
)

print("Remaining completed matches:", rows_after_status_filter)

Matches removes because they were not completed: 972
Remaining completed matches: 26602


Now I remove rows missing essential information. 
Our first models will need:

the two players;
the date;
rankings;
ranking points.

A match without this information cannot currently be represented consistently.

We are not removing rows because any arbitrary value is missing. Betting odds are allowed to remain missing because we have not decided to use them yet.

In [18]:
rows_before_missing_filter = len(matches)

matches = matches.dropna(
    subset = essential_columns
).copy()

rows_after_missing_filter = len(matches)

print(
    "Matches removed because essential values were missing",
    rows_before_missing_filter - rows_after_missing_filter
)

print("Remaining matches:", rows_after_missing_filter)

Matches removed because essential values were missing 66
Remaining matches: 26536


Check for invalid player records: 
If the result is zero i can continue. 

In [19]:
same_player_rows = matches["Winner"].eq(matches["Loser"])

print(
    "Rows where winner and loser are the same player:",
    same_player_rows.sum()
)

Rows where winner and loser are the same player: 0


Check if duplicate rows exists. 

In [20]:
duplicate_count = matches.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


Check if the same match is present more than once.
(Maybe the two rows were not exactly equal, but the informations that identify the match are.)

In [22]:
match_identifier_columns = [
    "Date", 
    "Tournament",
    "Winner",
    "Loser"
]

possible_duplicates = matches.duplicated( 
    subset = match_identifier_columns, 
    keep = False
)

print(
    "Rows blonging to possible duplicated matches:",
    possible_duplicates.sum()
)


Rows blonging to possible duplicated matches: 0


In [23]:
## Order matches chronologically

matches = matches.sort_values (
    by = [
        "Date", 
        "ATP",
        "Tournament", 
        "Winner",
        "Loser"
    ]
).reset_index(drop=True)

matches[["Date", "Tournament", "Winner", "Loser"]].head(10)

,Date,Tournament,Winner,Loser
0,2015-01-05,Brisbane International,Chardy J.,Golubev A.
1,2015-01-05,Brisbane International,Duckworth J.,Simon G.
2,2015-01-05,Brisbane International,Kokkinakis T.,Benneteau J.
3,2015-01-05,Brisbane International,Tomic B.,Querrey S.
4,2015-01-05,Chennai Open,Coric B.,Haase R.
5,2015-01-05,Chennai Open,Muller G.,Roger-Vasselin E.
6,2015-01-05,Qatar Exxon Mobil Open,Bolelli S.,Becker B.
7,2015-01-05,Qatar Exxon Mobil Open,Brown D.,Lorenzi P.
8,2015-01-05,Qatar Exxon Mobil Open,Dodig I.,Safwat M.
9,2015-01-05,Qatar Exxon Mobil Open,Gasquet R.,Andujar P.


I have to create a reproducible neutral player order. 
This means that every time you run the notebook with the same rows and ordering, the same players will be assigned to Player 1.

Fpr each match I generate a random decimal between 0 and 1. The comparison (< 0.5) turns each decimal into either: True or False. Approximately half of the values will be True. When the value is True, Player 1 will be the original winner. When the value is False, Player 1 will be the original loser. 

In [24]:
random_generator = np.random.default_rng(seed = 42)

player1_is_original_winner = (
    random_generator.random(len(matches)) < 0.5
)

I define as context_columns all the features that describe the ,atch but do not depend in who won. 

In [25]:
context_columns = [
    "ATP",
    "Location",
    "Tournament",
    "Date",
    "Series",
    "Court",
    "Surface",
    "Round",
    "Best of",
    "SourceYear"  
]

neutral_matches = matches[context_columns].copy()

In [26]:
neutral_matches["Player1"] = np.where(
    player1_is_original_winner, 
    matches["Winner"], 
    matches["Loser"]
)

neutral_matches["Player2"] = np.where(
    player1_is_original_winner, 
    matches["Loser"], 
    matches["Winner"]
)

In [27]:
neutral_matches["Player1Rank"] = np.where(
    player1_is_original_winner,
    matches["WRank"], 
    matches["LRank"]
)

neutral_matches["Player2Rank"] = np.where(
    player1_is_original_winner,
    matches["LRank"],
    matches["WRank"]
)

neutral_matches["Player1Points"] = np.where(
    player1_is_original_winner,
    matches["WPts"],
    matches["LPts"]
)

neutral_matches["Player2Points"] = np.where(
    player1_is_original_winner,
    matches["LPts"],
    matches["WPts"]
)

neutral_matches["Player1B365Odds"] = np.where(
    player1_is_original_winner,
    matches["B365W"],
    matches["B365L"]
)

neutral_matches["Player2B365Odds"] = np.where(
    player1_is_original_winner,
    matches["B365L"],
    matches["B365W"]
)

neutral_matches["Player1PSOdds"] = np.where(
    player1_is_original_winner,
    matches["PSW"],
    matches["PSL"]
)

neutral_matches["Player2PSOdds"] = np.where(
    player1_is_original_winner,
    matches["PSL"],
    matches["PSW"]
)

neutral_matches["Player1MaxOdds"] = np.where(
    player1_is_original_winner,
    matches["MaxW"],
    matches["MaxL"]
)

neutral_matches["Player2MaxOdds"] = np.where(
    player1_is_original_winner,
    matches["MaxL"],
    matches["MaxW"]
)

neutral_matches["Player1AvgOdds"] = np.where(
    player1_is_original_winner,
    matches["AvgW"],
    matches["AvgL"]
)

neutral_matches["Player2AvgOdds"] = np.where(
    player1_is_original_winner,
    matches["AvgL"],
    matches["AvgW"]
)

Create target. Convert true=1, false=0. 
If Player1Won = 0 it means that Player2 won. 

In [28]:
neutral_matches["Player1Won"] = (
    player1_is_original_winner.astype(int)
)

add a match identifier and organize the columns. 

In [30]:
neutral_matches.insert(
    0, 
    "MatchID",
    np.arange(len(neutral_matches))
)

In [31]:
ordered_columns = [
    "MatchID",
    "Date",
    "SourceYear",
    "ATP",
    "Location",
    "Tournament",
    "Series",
    "Court",
    "Surface",
    "Round",
    "Best of",
    "Player1",
    "Player2",
    "Player1Rank",
    "Player2Rank",
    "Player1Points",
    "Player2Points",
    "Player1B365Odds",
    "Player2B365Odds",
    "Player1PSOdds",
    "Player2PSOdds",
    "Player1MaxOdds",
    "Player2MaxOdds",
    "Player1AvgOdds",
    "Player2AvgOdds",
    "Player1Won"   
]

neutral_matches = neutral_matches[ordered_columns]
neutral_matches.head(10)


,MatchID,Date,SourceYear,ATP,Location,Tournament,Series,Court,Surface,Round,...,Player2Points,Player1B365Odds,Player2B365Odds,Player1PSOdds,Player2PSOdds,Player1MaxOdds,Player2MaxOdds,Player1AvgOdds,Player2AvgOdds,Player1Won
0,0,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1195.0,3.50,1.28,3.50,1.34,3.50,1.36,3.30,1.32,0
1,1,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1730.0,4.50,1.18,4.67,1.23,4.73,1.23,4.31,1.20,1
2,2,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,341.0,1.44,2.62,1.53,2.67,1.53,2.80,1.47,2.62,0
3,3,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,797.0,2.25,1.57,2.37,1.65,2.37,1.67,2.25,1.61,0
4,4,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,620.0,1.72,2.00,1.75,2.18,1.80,2.25,1.72,2.07,1
5,5,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,855.0,2.10,1.66,2.10,1.81,2.15,1.81,2.05,1.73,0
6,6,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,810.0,1.61,2.20,1.76,2.16,1.76,2.25,1.67,2.15,0
7,7,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,549.0,2.37,1.53,2.55,1.57,2.62,1.60,2.40,1.54,0
8,8,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,168.0,1.16,5.00,1.17,5.84,1.19,5.84,1.16,5.05,1
9,9,2015-01-05,2015,3,Doha,Qatar Exxon Mobil Open,ATP250,Outdoor,Hard,1st Round,...,950.0,1.20,4.33,1.24,4.44,1.26,4.84,1.21,4.15,1


Verify that the transformation is correct.
The code reconstructs the winner from the transformed table:

when Player1Won = 1, the winner should be Player 1;
when Player1Won = 0, the winner should be Player 2.

It then compares the reconstructed winner with the original Winner column.

If the result is True, every row was mapped consistently.

In [32]:
reconstructed_winner = np.where(
    neutral_matches["Player1Won"].eq(1),
    neutral_matches["Player1"],
    neutral_matches["Player2"]
)

transformation_is_correct = (
    reconstructed_winner == matches["Winner"].to_numpy()
).all()

print(
    "Every winner was reconstructed correctly.",
    transformation_is_correct
)

Every winner was reconstructed correctly. True


In [34]:
assert neutral_matches["Player1"].ne(
    neutral_matches["Player2"]
).all()

assert set(
    neutral_matches["Player1Won"].unique()
) == {0, 1}

print("Basic validation tests passed.")

Basic validation tests passed.


Check the target balance, it shoul be approximately 50 50. 

In [35]:
target_counts = neutral_matches["Player1Won"].value_counts()
target_percentages = (
    neutral_matches["Player1Won"]
    .value_counts(normalize = True)
    .sort_index()
    .mul(100)
)

print("Target counts:")
print(target_counts)

print("\nTarget percentages:")
print(target_percentages)

Target counts:
Player1Won
0    13334
1    13202
Name: count, dtype: int64

Target percentages:
Player1Won
0    50.248719
1    49.751281
Name: proportion, dtype: float64


In [36]:
output_file = (
    PROCESSED_DATA_DIR
    / "matches_neutral_2015_2025.csv"
)

neutral_matches.to_csv(
    output_file,
    index=False ##it prevents from saving the row index as an unnecessary CSV column
)

print("Dataset saved to:")
print(output_file.resolve())

Dataset saved to:
C:\Users\User\Downloads\artificial_project\data\processed\matches_neutral_2015_2025.csv


In [37]:
##verify that the file can be used later 

saved_matches = pd.read_csv(
    output_file, 
    parse_dates=["Date"] ## restore date as a date instead of ordinary text
)

print("Saved dataset shape:", saved_matches.shape)
print("First date: ", saved_matches["Date"].min())
print("Last date", saved_matches["Date"].max())

saved_matches.head()



Saved dataset shape: (26536, 26)
First date:  2015-01-05 00:00:00
Last date 2025-11-16 00:00:00


,MatchID,Date,SourceYear,ATP,Location,Tournament,Series,Court,Surface,Round,...,Player2Points,Player1B365Odds,Player2B365Odds,Player1PSOdds,Player2PSOdds,Player1MaxOdds,Player2MaxOdds,Player1AvgOdds,Player2AvgOdds,Player1Won
0,0,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1195.0,3.50,1.28,3.50,1.34,3.50,1.36,3.30,1.32,0
1,1,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1730.0,4.50,1.18,4.67,1.23,4.73,1.23,4.31,1.20,1
2,2,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,341.0,1.44,2.62,1.53,2.67,1.53,2.80,1.47,2.62,0
3,3,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,797.0,2.25,1.57,2.37,1.65,2.37,1.67,2.25,1.61,0
4,4,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,620.0,1.72,2.00,1.75,2.18,1.80,2.25,1.72,2.07,1


In [38]:
print("Final shape:", neutral_matches.shape)
print(neutral_matches["Player1Won"].value_counts(normalize=True))
print("First date:", neutral_matches["Date"].min())
print("Last date", neutral_matches["Date"].max())


Final shape: (26536, 26)
Player1Won
0    0.502487
1    0.497513
Name: proportion, dtype: float64
First date: 2015-01-05 00:00:00
Last date 2025-11-16 00:00:00
